# 第10回：モデル対決

**今日の問い：複雑なモデルは本当にいつも優れているか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

        - 同じ分割・指標で複数モデルを比較する
- 性能・速度・説明性・安定性を合わせて評価する
- 学習と検証の差から過学習を読む

        ### 進み方

        `CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
        `SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

        ### 先に押さえる言葉

        - 線形モデル：特徴量効果を重みの和で表すモデル
- 決定木：条件分岐を重ねるモデル
- アンサンブル：複数モデルを組み合わせる方法
- 交差検証：分割を変えて性能の安定性を見る方法

        > **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["active"], test_size=0.25, random_state=42, stratify=df["active"])
models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)),
    "Tree": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=4, random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)),
    "Gradient Boosting": make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingClassifier(max_iter=100, random_state=42)),
}


## TRY：同じ分割・同じ指標で比較


In [ ]:
rows=[]
for name, model in models.items():
    start=time.perf_counter()
    model.fit(X_train, y_train)
    elapsed=time.perf_counter()-start
    rows.append({"モデル": name, "検証F1": f1_score(y_valid, model.predict(X_valid)), "学習秒": elapsed})
comparison=pd.DataFrame(rows).sort_values("検証F1", ascending=False)
comparison.round({"検証F1": 3, "学習秒": 4})


## 5人の担当案

1. Dummy：単純基準
2. Logistic：説明しやすい線形モデル
3. Tree：1本の決定木
4. Random Forest：複数の木
5. Gradient Boosting：前の誤りを順に改善

スコアだけでなく、実行時間、説明しやすさ、安定性を1行ずつ共有します。


## DEEP DIVE：結果を一段深く読む

        次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

        ### 出力を見る観点

        - 1回の勝敗より平均とばらつきを見る
- わずかな改善と複雑化の釣り合いを考える
- 目的により最良モデルは変わる


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stability = []
for name, estimator in models.items():
    scores = cross_val_score(estimator, df[features], df["active"], cv=cv, scoring="f1")
    stability.append({"モデル": name, "F1平均": scores.mean(), "F1標準偏差": scores.std(), "最低F1": scores.min()})
display(pd.DataFrame(stability).sort_values("F1平均", ascending=False).round(3))


## よくある誤り

        - 異なる分割で比較する
- モデルごとに異なる指標を報告する
- 最も高い1回のスコアだけを採用する

        ## SELF-STUDY（任意・30〜60分）

        - 5-fold CVでF1平均・標準偏差・時間を比較する
- 説明重視と性能重視の2用途で推奨モデルを選ぶ

        成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

        ## 振り返りチェック

        1. 公平な比較に固定すべきものは何か
2. ばらつきが大きいモデルをどう扱うか
3. 最高スコア以外の選択理由は何か

        答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
